# Distributed Mixture of Experts (MoE) & Expert Parallelism

## 1. Mathematical Foundations & Routing Mechanics
Mixture of Experts (MoE) replaces dense Feed-Forward Networks (FFN) with $E$ sparse subnetworks (experts). For an input token representation $x \in \mathbb{R}^d$, a gating/router network produces logits:

$$H(x) = x \cdot W_g$$

Top-$k$ routing selects the $k$ highest scoring experts:

$$\text{TopK Indices} = \operatorname{arg top-k}(H(x))$$
$$G(x)_i = \frac{\exp(H(x)_i)}{\sum_{j \in \text{TopK}} \exp(H(x)_j)}$$

The final output is the weighted sum of expert outputs:

$$y = \sum_{i \in \text{TopK}} G(x)_i \cdot E_i(x)$$

### The Load Balancing Dilemma & Auxiliary Loss
Without intervention, the router tends to route all tokens to the same small set of "popular" experts, starving other experts and causing severe GPU memory imbalance. To force uniform distribution, Switch Transformers and DeepSeek introduce an auxiliary load-balancing loss:

$$\mathcal{L}_{aux} = \alpha \cdot E \sum_{i=1}^E f_i \cdot P_i$$

Where:
- $f_i = \frac{1}{T} \sum_{t=1}^T \mathbb{I}(\text{token } t \text{ routed to expert } i)$ is the empirical fraction of tokens dispatched to expert $i$.
- $P_i = \frac{1}{T} \sum_{t=1}^T G(x_t)_i$ is the average gating probability assigned to expert $i$.
- Uniform routing minimizes $\mathcal{L}_{aux}$ when $f_i = P_i = \frac{1}{E}$.

In [ ]:
import numpy as np

class SparseMoERouter:
    """
    Simulates Top-2 Sparse MoE Router with Load Balancing Auxiliary Loss.
    """
    def __init__(self, num_experts=8, hidden_dim=32, top_k=2):
        self.num_experts = num_experts
        self.hidden_dim = hidden_dim
        self.top_k = top_k
        np.random.seed(42)
        self.W_g = np.random.randn(hidden_dim, num_experts) * 0.1

    def forward(self, x):
        # x: [num_tokens, hidden_dim]
        T = x.shape[0]
        logits = x @ self.W_g # [T, E]
        
        # Softmax over all experts for P_i calculation
        exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
        all_probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        
        # Top-k selection
        top_k_indices = np.argsort(-logits, axis=-1)[:, :self.top_k] # [T, k]
        
        # Re-normalized weights for selected top-k
        top_k_weights = np.zeros((T, self.top_k))
        for t in range(T):
            selected_exp = exp_logits[t, top_k_indices[t]]
            top_k_weights[t] = selected_exp / np.sum(selected_exp)
            
        # Compute Load Balancing Loss
        # f_i: fraction of tokens routed to expert i (count occurrences in top-k)
        counts = np.zeros(self.num_experts)
        for idx in top_k_indices.flatten():
            counts[idx] += 1
        f = counts / (T * self.top_k)
        
        # P_i: average probability across tokens
        P = np.mean(all_probs, axis=0)
        
        aux_loss = self.num_experts * np.sum(f * P)
        
        return top_k_indices, top_k_weights, aux_loss, f, P

# Simulation
num_tokens = 500
router = SparseMoERouter(num_experts=8, hidden_dim=32, top_k=2)
x_sample = np.random.randn(num_tokens, 32)
top_k_idx, top_k_w, aux_loss, f_vec, P_vec = router.forward(x_sample)

print(f"Processed Tokens: {num_tokens} | Active Top-k: 2 | Number of Experts: 8")
print(f"Auxiliary Balance Loss: {aux_loss:.4f} (Ideal uniform = 1.000)")
print("Token Dispatch Distribution across 8 Experts:")
for e_idx in range(8):
    bar = '#' * int(f_vec[e_idx] * 100)
    print(f"  Expert {e_idx}: {f_vec[e_idx]*100:5.1f}% tokens | Avg P: {P_vec[e_idx]:.3f} | {bar}")

## 2. Distributed All-to-All Token Dispatch Mechanics
In Expert Parallelism (EP), each GPU hosts a subset of experts (e.g., GPU 0 hosts Experts 0-1, GPU 1 hosts Experts 2-3). Tokens on GPU 0 destined for Expert 3 must be sent to GPU 1 via an `All-to-All` collective communication primitive.

In [ ]:
def simulate_all_to_all_dispatch(tokens_per_gpu=4, num_gpus=4, num_experts=8):
    """
    Demonstrates token sorting and All-to-All dispatch matrix.
    Each GPU has 2 experts: GPU 0 -> Exp 0,1; GPU 1 -> Exp 2,3; etc.
    """
    total_tokens = tokens_per_gpu * num_gpus
    # Randomly assign each token to an expert
    token_assignments = np.random.randint(0, num_experts, size=total_tokens)
    target_gpus = token_assignments // 2 # 2 experts per GPU
    
    # Build communication transfer matrix [source_gpu, target_gpu]
    comm_matrix = np.zeros((num_gpus, num_gpus), dtype=int)
    for t_idx, dest_gpu in enumerate(target_gpus):
        src_gpu = t_idx // tokens_per_gpu
        comm_matrix[src_gpu, dest_gpu] += 1
        
    return comm_matrix

np.random.seed(123)
matrix = simulate_all_to_all_dispatch(tokens_per_gpu=16, num_gpus=4, num_experts=8)
print("All-to-All Token Transfer Matrix (Rows=Src GPU, Cols=Dest GPU):")
print(matrix)
print("
Total tokens sent per GPU:", np.sum(matrix, axis=1))
print("Total tokens received per GPU:", np.sum(matrix, axis=0))

## 3. High-Yield Interview Questions & Edge Cases

### Q1: What is the Capacity Factor in MoE routing and what happens during Token Dropping?
**Answer:** The Capacity Factor ($C$) defines the maximum number of tokens an expert can accept in a batch:
$$\text{Expert Capacity} = \lceil \frac{T}{E} \times C \rceil$$
If more tokens than this capacity are routed to a single expert, the excess tokens are **dropped** (bypassed via residual connection directly without undergoing expert FFN transformation). A higher capacity factor ($C=1.25$ to $1.5$) reduces token dropping but wastes compute on padding when routing is uneven. Modern architectures like Mixtral and DeepSeek-V3 employ dynamic capacity or routing bias offsets to eliminate token dropping altogether.

### Q2: How does Expert Parallelism (EP) interact with Tensor Parallelism (TP) and Data Parallelism (DP)?
**Answer:** In 3D MoE parallelism (TP + EP + DP):
- **Tensor Parallelism** splits attention layers across GPUs within a single node over NVLink.
- **Expert Parallelism** distributes different FFN experts across different GPUs.
- **Data Parallelism** replicates the attention layers across nodes while distributing tokens.
During the forward pass, after TP attention produces token embeddings, an `All-to-All` communication rearranges tokens from Data-Parallel layout (tokens partitioned by batch) to Expert-Parallel layout (tokens partitioned by assigned expert). After expert computation, a second `All-to-All` restores original token batch ordering.